In [ ]:
!pip install huggingface_hub==0.25.0
!pip install -U transformers
!pip install -U datasets
!pip install -U accelerate
!pip install -U bitsandbytes
!pip install -U peft
!pip install trl==0.9.6

  Using cached huggingface_hub-0.25.0-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-0.25.0-py3-none-any.whl (436 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.25.0 which is incompatible.
diffusers 0.36.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.25.0 which is incompatible.
transformers 4.57.3 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 0.25.0 which is incompatible.
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
  Attempting uninstal

In [ ]:
# Import necessary libraries for the fine-tuning pipeline
import os
import torch
from datasets import load_dataset  # For loading the medical dataset
from transformers import (AutoModelForCausalLM,  # For loading pre-trained language models
                         AutoTokenizer,  # For tokenizing text data
                         BitsAndBytesConfig,  # For model quantization settings
                         TrainingArguments,  # For configuring training hyperparameters
                         logging)
from peft import LoraConfig, get_peft_model  # Parameter-Efficient Fine-Tuning with LoRA
from huggingface_hub import login  # For authenticating with HuggingFace
from trl import SFTTrainer, setup_chat_format  # For supervised fine-tuning and chat formatting
import bitsandbytes as bnb  # For 4-bit quantization to reduce memory usage

# Set up authentication with HuggingFace using a secure token from Kaggle secrets
hf_token = os.environ.get("hugging_face")
login(token=hf_token)

# Define the model and dataset to use
base_model = "google/gemma-2-2b-it"  # Starting with Google's Gemma 2 2B instruction-tuned model
new_model = "Gemma-2-2b-it-ChatDoctor"  # Name for our fine-tuned medical assistant model
dataset_name = "lavita/ChatDoctor-HealthCareMagic-100k"  # Medical Q&A dataset with 100k examples

# Configure hardware acceleration based on the GPU capabilities
if torch.cuda.get_device_capability()[0] >= 8:
   torch_dtype = torch.bfloat16  # Use bfloat16 precision on newer GPUs (Ampere architecture or newer)
   attn_implementation = "flash_attention_2"  # Use Flash Attention 2 for faster training on modern GPUs
else:
   torch_dtype = torch.float16  # Fall back to float16 on older GPUs
   attn_implementation = "eager"  # Use standard attention implementation on older hardware

# Configure model quantization settings to reduce memory requirements
bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,  # Load model in 4-bit precision instead of 16-bit to save memory
   bnb_4bit_quant_type="nf4",  # Use normalized float 4 quantization for better quality
   bnb_4bit_compute_dtype=torch_dtype,  # Use the precision determined by GPU capabilities
   bnb_4bit_use_double_quant=True,  # Apply double quantization for additional memory savings
)

In [ ]:
# Load the pre-trained model with quantization and hardware optimization settings
model = AutoModelForCausalLM.from_pretrained(
  base_model,                           # Use the Gemma 2 2B model we specified earlier
  quantization_config=bnb_config,       # Apply the 4-bit quantization configuration to reduce memory usage
  device_map="auto",                    # Automatically manage model placement across available GPUs/CPU
  attn_implementation=attn_implementation  # Use the attention implementation we selected based on GPU capabilities
)

# Load the tokenizer for the same model
tokenizer = AutoTokenizer.from_pretrained(
  base_model,                           # Use the tokenizer that matches our base model
  trust_remote_code=True                # Allow running remote code in the tokenizer implementation for full functionality
)

# This function identifies all linear layers in the model that should be modified with LoRA
def find_all_linear_names(model):
  cls = bnb.nn.Linear4bit                # Look specifically for 4-bit quantized linear layers
  lora_module_names = set()              # Create an empty set to store eligible layer names

  # Iterate through all named modules in the model
  for name, module in model.named_modules():
      if isinstance(module, cls):        # Check if the module is a 4-bit linear layer
          names = name.split('.')        # Split the full path name into components
          if len(names) == 1:            # If it's a top-level module
              lora_module_names.add(names[0])
          else:                          # If it's a nested module
              lora_module_names.add(names[-1])  # Add only the base name of the module

  lora_module_names.discard('lm_head')   # Remove the language model head, which shouldn't be modified with LoRA
  return list(lora_module_names)         # Return the list of eligible layer names

# Get the list of modules that will be fine-tuned using LoRA
modules = find_all_linear_names(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
   r=16,
   lora_alpha=32,
   lora_dropout=0.05,
   bias="none",
   task_type="CAUSAL_LM",
   target_modules=modules
)

tokenizer.chat_template = None
model, tokenizer = setup_chat_format(model, tokenizer)
model = get_peft_model(model, peft_config)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
import re
from datasets import load_dataset

dataset = load_dataset(
   dataset_name,
   split="all",
   cache_dir="./cache"
)

dataset = dataset.shuffle(seed=42).select(range(2000))

def clean_text(text):
   text = re.sub(r'\b(?:www\.[^\s]+|http\S+)', '', text)
   text = re.sub(r'\b(?:aCht Doctor(?:.com)?(?:.in)?|www\.(?:google|yahoo)\S*)', '', text)
   text = re.sub(r'\s+', ' ', text)
   return text.strip()

def format_chat_template(row):
   cleaned_instruction = clean_text(row["instruction"])  # Очистка инструкции
   cleaned_input = clean_text(row["input"])             # Очистка входных данных
   cleaned_output = clean_text(row["output"])           # Очистка выходных данных

   row_json = [
       {"role": "system", "content": cleaned_instruction},
       {"role": "user", "content": cleaned_input},
       {"role": "assistant", "content": cleaned_output}
   ]
   row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
   return row

dataset = dataset.map(format_chat_template, num_proc=4)
dataset = dataset.train_test_split(test_size=0.1)
data_collator = lambda batch: tokenizer(
   batch["text"],
   return_tensors="pt",
   padding=True,
   truncation=True
)

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=10,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    fp16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=512,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:408: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:413: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


TrainOutput(global_step=339, training_loss=2.36630739082629, metrics={'train_runtime': 4006.5063, 'train_samples_per_second': 1.348, 'train_steps_per_second': 0.085, 'total_flos': 1.696109529153024e+16, 'train_loss': 2.36630739082629, 'epoch': 3.0})

In [ ]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained(new_model)
merged_model.push_to_hub(new_model, use_temp_dir=False)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tDoctor/model.safetensors:   1%|          | 21.0MB / 2.22GB            

CommitInfo(commit_url='https://huggingface.co/Xasan01/Gemma-2-2b-it-ChatDoctor/commit/13cd5331e6e2625ec05b672dc7047f04b04e4214', commit_message='Upload Gemma2ForCausalLM', commit_description='', oid='13cd5331e6e2625ec05b672dc7047f04b04e4214', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Xasan01/Gemma-2-2b-it-ChatDoctor', endpoint='https://huggingface.co', repo_type='model', repo_id='Xasan01/Gemma-2-2b-it-ChatDoctor'), pr_revision=None, pr_num=None)

In [ ]:
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('./final_model/tokenizer_config.json',
 './final_model/special_tokens_map.json',
 './final_model/chat_template.jinja',
 './final_model/tokenizer.model',
 './final_model/added_tokens.json',
 './final_model/tokenizer.json')

In [ ]:
from peft import PeftModel

base_model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained("./final_model")

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(model, "./final_model")

model = model.merge_and_unload()

test_prompt = """<|im_start|>system
If you are a doctor, please answer the medical questions based on the patient's description.<|im_end|>
<|im_start|>user
I have a headache and fever for 2 days. What should I do?<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(response)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

<|im_start|><|im_start|>system
If you are a doctor, please answer the medical questions based on the patient's description.<|im_end|>
<|im_start|>user
I have a headache and fever for 2 days. What should I do?<|im_end|>
<|im_start|>assistant
It sounds like you have a bad headache with a fever for two days.  Here is what you can do:

**1. Rest:**
   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *   *


In [ ]:
# ! pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 17.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4422291 sha256=c636f2c5f6ad8719b1a8de98b76a9c13338e433ef9eb22305360d4ecbe2acdc7
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [ ]:
# # OLLAMA  ga ulash
# tokenizer.save_pretrained("gemma-medical-merged")


('gemma-medical-merged/tokenizer_config.json',
 'gemma-medical-merged/special_tokens_map.json',
 'gemma-medical-merged/chat_template.jinja',
 'gemma-medical-merged/tokenizer.model',
 'gemma-medical-merged/added_tokens.json',
 'gemma-medical-merged/tokenizer.json')

In [ ]:
# # Save directly as GGUF format
# merged_model.save_pretrained_gguf(
#     "gguf_model",
#     tokenizer,
#     quantization_method="q4_k_m"
# )

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# local_save_path = "/content/LLM_medicina_done"

# # 2. Model va tokennayzerni shu yerga saqlash
# merged_model.save_pretrained(local_save_path)
# tokenizer.save_pretrained(local_save_path)

# print("Model va tokennayzer Colab local xotirasiga saqlandi!")

Model va tokennayzer Colab local xotirasiga saqlandi!


In [ ]:
# # Drive-da yangi papka yaratish
# !mkdir -p /content/drive/MyDrive/LLM_medicina_backup

# # Local papkadagi hamma narsani Drive-ga nusxalash
# !cp -r /content/LLM_medicina_done/* /content/drive/MyDrive/LLM_medicina_backup/

# print("Model muvaffaqiyatli Drive-ga nusxalandi! ✅")

Model muvaffaqiyatli Drive-ga nusxalandi! ✅


In [ ]:
# !ls -lh /content/drive/MyDrive/LLM_medicina_backup

total 2.2G
-rw------- 1 root root   53 Dec 27 12:44 added_tokens.json
-rw------- 1 root root  196 Dec 27 12:44 chat_template.jinja
-rw------- 1 root root 1.9K Dec 27 12:44 config.json
-rw------- 1 root root  193 Dec 27 12:44 generation_config.json
-rw------- 1 root root 2.1G Dec 27 12:45 model.safetensors
-rw------- 1 root root  647 Dec 27 12:45 special_tokens_map.json
-rw------- 1 root root  46K Dec 27 12:45 tokenizer_config.json
-rw------- 1 root root  33M Dec 27 12:45 tokenizer.json
-rw------- 1 root root 4.1M Dec 27 12:45 tokenizer.model


In [ ]:
# from google.colab import files
# import os

# # Folder where GGUF model is saved
# gguf_dir = "gguf_model"

# # Find GGUF files
# gguf_files = [f for f in os.listdir(gguf_dir) if f.endswith(".gguf")]

# if gguf_files:
#     gguf_file = os.path.join(gguf_dir, gguf_files[0])
#     print("Downloading:", gguf_file)
#     files.download(gguf_file)
# else:
#     print("No GGUF file found.")


In [ ]:
# !git clone https://github.com/ggerganov/llama.cpp
# !pip install -r llama.cpp/requirements.txt

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# import os

# # Drive-dagi papkangiz nomi
# path = "/content/drive/MyDrive/LLM_medicina_backup"

# if os.path.exists(path):
#     print("✅ Papka topildi!")
#     print("Fayllar ro'yxati:", os.listdir(path))
# else:
#     print("❌ Papka hali ham topilmadi. Drive-dagi 'Mening diskim' (My Drive) bo'limida 'LLM_medicina_backup' nomli papka borligini tekshiring.")

✅ Papka topildi!
Fayllar ro'yxati: ['added_tokens.json', 'chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer_config.json', 'tokenizer.json', 'tokenizer.model']


In [ ]:
# # 1. llama.cpp vositasini yuklab olish va sozlash
# !git clone https://github.com/ggerganov/llama.cpp
# !pip install -r llama.cpp/requirements.txt

# # 2. Konvertatsiya qilish
# !python llama.cpp/convert_hf_to_gguf.py "/content/drive/MyDrive/LLM_medicina_backup" \
#   --outfile "/content/drive/MyDrive/LLM_medicina_backup/medical_f16.gguf" \
#   --outtype f16

In [ ]:
# from unsloth import FastLanguageModel
# import torch

# # Drive-dagi saqlangan model manzili
# model_path = "/content/drive/MyDrive/LLM_medicina_backup"

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = model_path, # Saqlangan papkani ko'rsatamiz
#     max_seq_length = 2048,
#     load_in_4bit = True,
# )
# FastLanguageModel.for_inference(model) # Modelni tayyorlash

==((====))==  Unsloth 2025.12.9: Fast Gemma2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/content/drive/MyDrive/LLM_medicina_backup does not have a padding token! Will use pad_token = <pad>.


Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256002, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
          (rotary_emb): GemmaFixedRotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Ge

In [ ]:
# import os

# # 1. Avval papkani yaratib olamiz
# temp_path = "/content/medical_gguf_temp"
# if not os.path.exists(temp_path):
#     os.makedirs(temp_path)
#     print(f"✅ Papka yaratildi: {temp_path}")

# # 2. Endi GGUF-ga eksport qilamiz
# try:
#     model.save_pretrained_gguf(
#         temp_path,
#         tokenizer,
#         quantization_method = "q4_k_m"
#     )
#     print("🚀 GGUF fayli muvaffaqiyatli yaratildi!")
# except Exception as e:
#     print(f"❌ Xatolik yuz berdi: {e}")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


✅ Papka yaratildi: /content/medical_gguf_temp
Unsloth: Merging model weights to 16-bit format...
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
❌ Xatolik yuz berdi: Unsloth: GGUF conversion failed: Unsloth: `config.json` does not exist inside `/content/medical_gguf_temp`.
